# Chapter 1 — AI Foundations & Text Processing: Pattern Recognition and NLP Tasks

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winnertechserv/aimlgybs/blob/main/part1_foundations/ch01_ai_foundations_text_processing/ch01_notebook.ipynb)

Companion notebook for **Chapter 1** of *AI-Machine Learning: Get Your Base Strong*. Read the chapter first, then run each cell and compare your output with the output printed in the book (shown below each cell).


In [ ]:
# Optional: install the exact library versions used for the outputs in the book.
# Google Colab already has these libraries; newer versions may change a few digits.
# !pip install -q numpy==2.5.3 pandas==3.0.5 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 pillow==12.3.0

## Key takeaways

- AI can be defined as acting humanly (the Turing Test), thinking rationally, or automating tasks that need intelligence.
- Machine learning is pattern recognition from examples. The goal is good generalization to unseen data.
- NLP tasks are NLU (understanding), NLG (generating), or both: question answering, summarization and translation.
- Sentiment analysis needs context: word order and negation can flip the meaning of the same words.
- POS tagging labels grammar; NER labels real-world things such as people, places, organizations and dates.
- Extractive summaries pick original sentences; abstractive summaries write new ones.
- Translation is hard because languages differ in word forms, grammar and word order.

## Example 1: Pattern Recognition — Predicting the Next Number

This tiny 'learner' looks at a sequence and tries several kinds of rule: multiply by a constant, add up the previous terms, or a polynomial found by taking differences until they become constant. It keeps whichever rule fits and uses it to predict the next number. Look at the last one: 3, 5, 12, 24, 41 has differences 2, 7, 12, 17, which grow by 5 each time, so the next term is 63.

In [1]:
# Chapter 1 - Example 1: Pattern recognition - finding the rule behind a sequence
# A tiny "learner" tries a few kinds of rule and keeps the one that fits the data.
import numpy as np

def find_rule(seq):
    s = np.array(seq, dtype=float)
    # Rule 1: constant ratio (geometric, e.g. 1, 2, 4, 8)
    r = s[1:] / s[:-1]
    if np.allclose(r, r[0]):
        return f"multiply by {r[0]:g}", s[-1] * r[0]
    # Rule 2: each term = sum of previous k terms (Fibonacci-like)
    for k in (2, 3):
        if all(s[i] == s[i - k:i].sum() for i in range(k, len(s))):
            return f"sum of previous {k} terms", s[-k:].sum()
    # Rule 3: polynomial - take differences until they become constant
    diffs, level = [s], 0
    while len(diffs[-1]) > 1 and not np.allclose(diffs[-1], diffs[-1][0]):
        diffs.append(np.diff(diffs[-1])); level += 1
    if len(diffs[-1]) > 1:
        nxt = diffs[-1][0]
        for d in reversed(diffs[:-1]):
            nxt = d[-1] + nxt
        return f"polynomial of degree {level}", nxt
    # Rule 4: primes
    def is_prime(k): return k > 1 and all(k % p for p in range(2, int(k**0.5) + 1))
    if all(is_prime(int(v)) for v in s):
        k = int(s[-1]) + 1
        while not is_prime(k): k += 1
        return "next prime number", k
    return "no rule found", None

sequences = [
    [1, 2, 3, 4, 5],
    [1, 3, 5, 7, 9],
    [2, 3, 5, 7, 11],
    [1, 4, 9, 16, 25],
    [1, 2, 4, 8, 16, 32],
    [1, 1, 2, 3, 5, 8],
    [1, 1, 2, 4, 7, 13],
    [3, 5, 12, 24, 41],
]
for seq in sequences:
    rule, nxt = find_rule(seq)
    print(f"{str(seq):24s} rule: {rule:26s} next -> {nxt:g}")

[1, 2, 3, 4, 5]          rule: polynomial of degree 1     next -> 6
[1, 3, 5, 7, 9]          rule: polynomial of degree 1     next -> 11
[2, 3, 5, 7, 11]         rule: next prime number          next -> 13
[1, 4, 9, 16, 25]        rule: polynomial of degree 2     next -> 36
[1, 2, 4, 8, 16, 32]     rule: multiply by 2              next -> 64
[1, 1, 2, 3, 5, 8]       rule: sum of previous 2 terms    next -> 13
[1, 1, 2, 4, 7, 13]      rule: sum of previous 3 terms    next -> 24
[3, 5, 12, 24, 41]       rule: polynomial of degree 2     next -> 63


## Example 2: Sentiment Analysis with TF-IDF and Logistic Regression

TF-IDF turns each review into a vector of word weights, and logistic regression learns which words point to 'positive' and which to 'negative'. We also include word pairs (ngram_range=(1, 2)). That lets the model learn 'not good' and 'not bad' as separate clues, so it correctly separates the two tricky sentences from the overview. The last lines show the strongest signals the model learned for each class.

In [2]:
# Chapter 1 - Example 2: Text classification (sentiment analysis)
# Words -> numbers with TF-IDF, then a classifier learns which words signal which mood.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

train_texts = [
    "the food was good", "really good service and tasty food", "great place loved it",
    "the staff were friendly and helpful", "excellent meal will come again",
    "not bad at all, quite good", "the pizza was delicious", "lovely ambience and good music",
    "the food was bad", "terrible service and cold food", "awful place hated it",
    "the staff were rude and slow", "worst meal ever never again",
    "not good at all, quite bad", "the pizza was burnt", "noisy and dirty, very bad",
]
train_labels = ["pos"] * 8 + ["neg"] * 8

# ngram_range=(1,2) also keeps word PAIRS like "not good" and "not bad"
vec = TfidfVectorizer(ngram_range=(1, 2))
X = vec.fit_transform(train_texts)
clf = LogisticRegression(C=10, max_iter=1000).fit(X, train_labels)
print("Vocabulary size (words + word pairs):", len(vec.vocabulary_))

tests = ["The food was good, not bad at all",
         "The food was bad, not good at all",
         "friendly staff and delicious pizza",
         "rude staff and cold pizza"]
for t in tests:
    p = clf.predict_proba(vec.transform([t]))[0]
    label = clf.classes_[p.argmax()]
    print(f"{t:38s} -> {label}  (P(pos) = {p[list(clf.classes_).index('pos')]:.2f})")

# Which features push hardest toward each class?
names = vec.get_feature_names_out()
w = clf.coef_[0]            # positive weight -> class 'pos' (classes_ = ['neg','pos'])
order = w.argsort()
print("\nStrongest NEGATIVE signals:", [names[i] for i in order[:5]])
print("Strongest POSITIVE signals:", [names[i] for i in order[-5:][::-1]])

Vocabulary size (words + word pairs): 98
The food was good, not bad at all      -> pos  (P(pos) = 0.73)
The food was bad, not good at all      -> neg  (P(pos) = 0.27)
friendly staff and delicious pizza     -> pos  (P(pos) = 0.72)
rude staff and cold pizza              -> neg  (P(pos) = 0.27)

Strongest NEGATIVE signals: ['bad', 'was bad', 'was burnt', 'burnt', 'not good']
Strongest POSITIVE signals: ['good', 'was good', 'delicious', 'was delicious', 'quite good']


## Example 3: Extractive Summarization

This is the simplest extractive summarizer. It counts how often each meaningful word appears (ignoring filler words such as 'the'), scores each sentence by the average importance of its words, and keeps the top three in their original order. Notice that the off-topic sentence about the weather gets a low score and is left out.

In [3]:
# Chapter 1 - Example 3: Extractive text summarization
# Score each sentence by how many important (frequent, non-filler) words it contains,
# then keep the top sentences in their original order.
import re
from collections import Counter

document = """Machine translation converts text from one language into another.
It is hard because languages differ in word order and grammar.
German, for example, usually puts the verb in a different place from Japanese.
Modern translation systems learn from millions of translated sentence pairs.
The weather was pleasant on the day the first system was demonstrated.
These systems use neural networks that read the whole sentence before translating it.
Good translation must keep both the meaning and the style of the original text."""

STOP = set("""a an the is it in of from to and or on was were be that this for
by with must both before its one into usually""".split())

sentences = [s.strip() for s in document.split("\n") if s.strip()]
words = [w for w in re.findall(r"[a-z]+", document.lower()) if w not in STOP]
freq = Counter(words)
top = max(freq.values())
weight = {w: c / top for w, c in freq.items()}          # normalize to 0..1
print("Most important words:", [w for w, _ in freq.most_common(6)])

scores = []
for i, s in enumerate(sentences):
    toks = [w for w in re.findall(r"[a-z]+", s.lower()) if w not in STOP]
    scores.append((sum(weight[w] for w in toks) / max(len(toks), 1), i))

print("\nSentence scores:")
for sc, i in scores:
    print(f"  {sc:.2f}  {sentences[i][:62]}")

k = 3
chosen = sorted(i for _, i in sorted(scores, reverse=True)[:k])
print(f"\nSummary ({k} of {len(sentences)} sentences):")
for i in chosen:
    print(" -", sentences[i])

Most important words: ['translation', 'text', 'systems', 'sentence', 'machine', 'converts']

Sentence scores:
  0.50  Machine translation converts text from one language into anoth
  0.33  It is hard because languages differ in word order and grammar.
  0.33  German, for example, usually puts the verb in a different plac
  0.50  Modern translation systems learn from millions of translated s
  0.33  The weather was pleasant on the day the first system was demon
  0.41  These systems use neural networks that read the whole sentence
  0.48  Good translation must keep both the meaning and the style of t

Summary (3 of 7 sentences):
 - Machine translation converts text from one language into another.
 - Modern translation systems learn from millions of translated sentence pairs.
 - Good translation must keep both the meaning and the style of the original text.
